# PC 2: Polynomial Interpolation


**<big><font color=black><span style="background-color:skyblue">To be submitted</span></font>: Exercise 2, Question 2</big>**

**Loading the Python modules**

In [ ]:
# Numerical arrays and elementary functions
import numpy as np

# Plotting
from matplotlib import pyplot as plt

## Notation and reminders

Throughout this practical class, we consider:

- a continuous function $f:[a,b]\rightarrow\mathbb{R}$;
- $\|f\|_\infty=\max_{x\in[a,b]}|f(x)|$;
- pairwise distinct points $x_0,\ldots,x_n$ in $[a,b]$;
- the Newton basis $\Pi_k(X)=\prod_{j=0}^{k-1}(X-x_j)$, with $\Pi_0=1$;
- the Lagrange basis $l_k(X)=\prod_{\substack{0\leq j\leq n\\j\neq k}}\frac{X-x_j}{x_k-x_j}$;
- $p_n(f)=\sum_{k=0}^n f(x_k)l_k$, the Lagrange interpolation polynomial of $f$ at the points $x_k$.

The **Lebesgue constant** $\Lambda_n$ is the norm of the Lagrange interpolation operator

$$
\mathcal L_n:\mathcal C^0([a,b])\longrightarrow\mathcal C^0([a,b]),
\qquad f\longmapsto p_n(f).
$$

Equivalently, for all $f,g\in\mathcal C^0([a,b])$,

$$
\|p_n(f)-p_n(g)\|_\infty\leq\Lambda_n\|f-g\|_\infty.
$$

The value of $\Lambda_n$ depends on the interpolation points. We shall use the following results:

- for equispaced points, $\Lambda_n\sim\frac{2^{n+1}}{en\log n}$;
- for Chebyshev points, $\frac{2}{\pi}\log(n+1)\leq\Lambda_n\leq\frac{2}{\pi}\log(n+1)+1$.

See references [1,2] for further details.

---

## Exercise 1: Error estimates and Runge's phenomenon

### I. A first error estimate

#### Question 1

Prove the following estimate. For every $f\in C^{n+1}([a,b])$ and every $x\in[a,b]$, there exists $\xi_x\in[a,b]$ such that

$$
f(x)-p_n(f)(x)=\frac{\Pi_{n+1}(x)}{(n+1)!}f^{(n+1)}(\xi_x).
$$

In particular,

$$
\|f-p_n(f)\|_\infty\leq\frac{\|\Pi_{n+1}\|_\infty}{(n+1)!}\|f^{(n+1)}\|_\infty.
$$

*Hint: Fix $x\in[a,b]$, introduce $p_{n+1,x}(f)$, the interpolation polynomial of $f$ at $(x_0,\ldots,x_n,x)$, and study $p_{n+1,x}(f)(X)-p_n(f)(X)$. *

#### Question 2

We shall use Cauchy's formula: if $f$ is analytic on an open set containing the closed disk with centre $x$ and radius $r$, then

$$
\frac{f^{(n)}(x)}{n!}=\frac{1}{2\pi}\int_0^{2\pi}\frac{f(x+re^{i\theta})e^{-in\theta}}{r^n}\,d\theta.
$$

Prove that if $f$ is analytic at $(a+b)/2$ with radius of convergence $R>3(b-a)/2$, then $\|f-p_n(f)\|_\infty\rightarrow0$ as $n\rightarrow\infty$.

#### Question 3

Deduce that, for equispaced interpolation points, the interpolation error for the sine function on $[-\pi,\pi]$ converges to zero faster than $\rho^n$ for every $\rho\in(0,1)$.

In the remainder of the notebook, we use the function in the next cell. It constructs the interpolation polynomial in the Lagrange basis,

$$
p_n(f)=\sum_{k=0}^n y_kl_k,
$$

where $y_k=f(x_k)$ and the Lagrange polynomials are defined above.

In [ ]:
def Lagrange(x_i, y_i, x):
    """
    Evaluate the interpolation polynomial associated with the data (x_i, y_i).

    Parameters
    ----------
    x_i : interpolation points, array of length N = n+1
    y_i : values at the interpolation points, array of length N = n+1
    x   : evaluation points, array of length Nx

    Returns
    -------
    Values of p_n(f) at x, as an array of length Nx.
    """
    N = len(x_i)
    Nx = len(x)

    # x_m_xi contains x-x_i for every evaluation point x and every i.
    # xi_m_xi contains x_i-x_j for every i and j.
    x_m_xi = x[:, np.newaxis] - x_i
    xi_m_xi = x_i - x_i[:, np.newaxis]
    li = np.zeros((Nx, N))
    for i in range(N):
        # Column i contains l_i(x) at every evaluation point.
        li[:, i] = np.prod(
            np.divide(x_m_xi[:, :i], xi_m_xi[np.newaxis, :i, i]), axis=1
        ) * np.prod(
            np.divide(x_m_xi[:, i+1:], xi_m_xi[np.newaxis, i+1:, i]), axis=1
        )

    return np.dot(li, y_i)

In [ ]:
# Example: use Lagrange interpolation for sin(x) on [-pi, pi].
def f1(x):
    return np.sin(x)

n = 50
Nx = 2000
xi = np.linspace(-np.pi, np.pi, n+1)
yi = f1(xi)
x = np.linspace(-np.pi, np.pi, Nx)

val = Lagrange(xi, yi, x)

plt.figure()
plt.plot(x, val, color="blue", label="Interpolation polynomial")
plt.scatter(xi, yi, color="red", label="Interpolation points")
plt.legend()
plt.show()

#### Question 4

We now verify this behaviour numerically. Continue to consider $f(x)=\sin x$ on $[-\pi,\pi]$. Approximate $\|f-p_n(f)\|_\infty$ by $\max_i|f(t_i)-p_n(f)(t_i)|$, where the $t_i$ are $N_t=2000$ equispaced points on $[-\pi,\pi]$.

1. Starting from the code below, plot $\max_i|f(t_i)-p_n(f)(t_i)|$ against $n$ for equispaced interpolation points and $2\leq n\leq30$.
2. Explain the behaviour for $n<20$ and suggest an interpretation for what happens around $n\approx20$.
3. On the same graph, plot the errors for $\sin x$ and $\sin(2x)$. Use the estimates from Question 1 to explain their different convergence rates.

In [ ]:
def f2(x):
    return np.sin(2*x)

# Compute the infinity-norm error against the interpolation degree n.
n = 30
Nt = 2000
t = np.linspace(-np.pi, np.pi, Nt)

tab_n = np.arange(2, n+1)
err_inf = np.zeros(n-1)     # Errors for sin(x)
err_inf2 = np.zeros(n-1)    # Errors for sin(2x)

for k in range(n-1):
    xi = np.linspace(-np.pi, np.pi, tab_n[k]+1)
    yi = f1(xi)
    yi2 = f2(xi)
    val = Lagrange(xi, yi, t)
    val2 = Lagrange(xi, yi2, t)
    err_inf[k] = max(abs(val-f1(t)))
    err_inf2[k] = max(abs(val2-f2(t)))

In [ ]:
# Plot the infinity-norm errors against the interpolation degree.
# Add your code here.


### II. A sharper estimate using the Lebesgue constant

#### Question 5

Prove that

$$
\|f-p_n(f)\|_\infty\leq(1+\Lambda_n)\inf_{Q\in\mathbb R_n[X]}\|f-Q\|_\infty.
$$

#### Question 6

Assume that, for every Lipschitz function $f$,

$$
\inf_{Q\in\mathbb R_n[X]}\|f-Q\|_\infty\leq\frac{C_f}{\sqrt n}.
$$

What can be deduced about the convergence of $p_n(f)$ to $f$ when the interpolation points are, respectively, equispaced points and Chebyshev points?

### III. Runge's phenomenon

#### Question 7

Consider the Runge function

$$
f_{\mathrm{Runge}}(x)=\frac{1}{1+25x^2}, \qquad x\in[-1,1]. \tag{1}
$$

1. In the cells below, observe $p_n(f)$ and its error for several values of $n$, first with equispaced interpolation points and then with Chebyshev points.
2. Use the estimates from Part II to explain the different convergence behaviour of the two choices.
3. Repeat the experiment with $\widetilde f_{\mathrm{Runge}}(x)=1/(1+0.1x^2)$. Explain why convergence is recovered for equispaced points.

In [ ]:
def fRunge(x):
    """Evaluate the Runge function at x."""
    return 1/(1+25*x*x)


def fRunge_tilde(x):
    """Evaluate the modified Runge function at x."""
    return 1/(1+0.1*x*x)

In [ ]:
def cheb_points(xmin, xmax, N):
    """
    Return N Chebyshev points in [xmin, xmax].

    Parameters
    ----------
    xmin, xmax : interval endpoints
    N          : number of required points
    """
    return (xmin+xmax)/2 + ((xmax-xmin)/2)*np.cos(
        ((2*np.arange(N)+1)*np.pi)/(2*N)
    )

In [ ]:
# Check cheb_points for two and three nodes.
N = 2
print(f"Chebyshev points for {N} nodes:", cheb_points(-1, 1, N))
N = 3
print(f"Chebyshev points for {N} nodes:", cheb_points(-1, 1, N))

In [ ]:
# Construct interpolation polynomials for the Runge function.
xmin = -1
xmax = 1
Nx = 1000
x = np.linspace(xmin, xmax, Nx)

# Polynomial degree
n = 10

# Equispaced points
xk_equi = np.linspace(xmin, xmax, n+1)
yk_equi = fRunge(xk_equi)
p_equi = Lagrange(xk_equi, yk_equi, x)

# Chebyshev points
xk_cheb = cheb_points(xmin, xmax, n+1)
yk_cheb = fRunge(xk_cheb)
p_cheb = Lagrange(xk_cheb, yk_cheb, x)

# Errors
err_equi = np.abs(fRunge(x)-p_equi)
err_cheb = np.abs(fRunge(x)-p_cheb)

In [ ]:
# Plot the results.
fig_equi = plt.figure()
plt.scatter(xk_equi, yk_equi, color="red", label="Data")
plt.plot(x, p_equi, color="green", label="P_n(x), equispaced points")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.title("Interpolation of the Runge function")
plt.show()

fig_cheb = plt.figure()
plt.scatter(xk_cheb, yk_cheb, color="red", label="Data")
plt.plot(x, p_cheb, color="green", label="P_n(x), Chebyshev points")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.show()

fig_err = plt.figure()
plt.plot(x, err_equi, color="green", label="Error: equispaced points")
plt.plot(x, err_cheb, color="purple", label="Error: Chebyshev points")
plt.xlabel("x")
plt.ylabel("Error")
plt.legend()
plt.show()

In [ ]:
# Construct interpolation polynomials for the modified Runge function.
xmin = -1
xmax = 1
Nx = 2000
x = np.linspace(xmin, xmax, Nx)

# Polynomial degree
n = 20

# Equispaced points
xk_equi = np.linspace(xmin, xmax, n+1)
yk_equi = fRunge_tilde(xk_equi)
p_equi = Lagrange(xk_equi, yk_equi, x)

# Chebyshev points
xk_cheb = cheb_points(xmin, xmax, n+1)
yk_cheb = fRunge_tilde(xk_cheb)
p_cheb = Lagrange(xk_cheb, yk_cheb, x)

# Errors
err_equi = np.abs(fRunge_tilde(x)-p_equi)
err_cheb = np.abs(fRunge_tilde(x)-p_cheb)

In [ ]:
# Plot the results.
fig_equi = plt.figure()
plt.scatter(xk_equi, yk_equi, color="red", label="Data")
plt.plot(x, p_equi, color="green", label="P_n(x), equispaced points")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.title("Interpolation of the modified Runge function")
plt.show()

fig_cheb = plt.figure()
plt.scatter(xk_cheb, yk_cheb, color="red", label="Data")
plt.plot(x, p_cheb, color="green", label="P_n(x), Chebyshev points")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.show()

fig_err = plt.figure()
plt.plot(x, err_equi, color="green", label="Error: equispaced points")
plt.plot(x, err_cheb, color="purple", label="Error: Chebyshev points")
plt.xlabel("x")
plt.ylabel("Error")
plt.legend()
plt.show()

---

## Exercise 2: Conditioning and stability of polynomial interpolation ([2,3])

For every continuous function $f$, define the condition number of the interpolation polynomial $p_n(f)$ by

$$
\kappa_n(f)=\frac{1}{\varepsilon}\sup_{\|\Delta f\|_\infty/\|f\|_\infty\leq\varepsilon}
\frac{\|p_n(f+\Delta f)-p_n(f)\|_\infty}{\|p_n(f)\|_\infty}.
$$

### I. Conditioning of polynomial interpolation and stability of the Lagrange formula

#### Question 1

Show that

$$
\kappa_n(f)\leq\frac{\|f\|_\infty}{\|p_n(f)\|_\infty}\Lambda_n.
$$

What does this imply about the conditioning of polynomial interpolation for different choices of interpolation points?

#### Question 2

**<font color=black><span style="background-color:skyblue">To be submitted</span></font>**

Consider again $f(x)=\sin x$ on $[-\pi,\pi]$. We study how perturbations of the sampled function affect its interpolation polynomial. Let $\widetilde f=f+\Delta f$, with $\|\Delta f\|_\infty\leq\varepsilon$ and $\varepsilon=2^{-10}$. At every interpolation point $x_i$, choose $\Delta f(x_i)$ independently and uniformly in $[-\varepsilon,\varepsilon]$.

1. Use 32 interpolation points, first equispaced and then Chebyshev points.
   - Plot both the interpolation polynomial of $f$ and the polynomial obtained from the perturbed data.
   - Explain the results in terms of conditioning and the value of $\Lambda_n$. What do they imply about the practical usefulness of interpolation at equispaced points?
2. For the unperturbed function, increase the number of Chebyshev points to approximately $n=660$.
   - Plot the interpolation polynomial and the target function.
   - Explain the results in terms of conditioning and stability. What is $\Lambda_n$ for this value of $n$?
   - Explain any error or warning messages.

In [ ]:
# Add as many code and Markdown cells as needed here to answer Question 2.


**<font color=black><span style="background-color:skyblue">End of submitted work</span></font>**

### II. Interpolation at Chebyshev points in the Chebyshev basis

Chebyshev points are the zeros of the Chebyshev polynomial $T_k$, where

$$
T_k(\cos\theta)=\cos(k\theta). \tag{2}
$$

#### Question 3

Consider the interpolation polynomial through $(x_k,y_k)_{k=0}^n$, where the $x_k$ are Chebyshev points, and expand it in the Chebyshev basis:

$$
p_n(f)=\sum_{k=0}^n c_kT_k.
$$

Write the linear system satisfied by the coefficients $c_k$.

#### Question 4

Using (2), show that the coefficients of the interpolation polynomial in the Chebyshev basis are

$$
c_0=\frac{1}{n+1}\sum_{i=0}^nT_0(x_i)y_i,
\qquad
c_j=\frac{2}{n+1}\sum_{i=0}^nT_j(x_i)y_i,
\quad j=1,\ldots,n.
$$

#### Question 5

The function `compute_Chebyshev_coefficients` below computes the coefficients $c_k$ of the interpolation polynomial in the Chebyshev basis. The function `evaluate_Chebyshev_interpolant` evaluates the resulting polynomial.

In [ ]:
def compute_Chebyshev_coefficients(xmin, xmax, yk):
    """
    Compute the coefficients in the Chebyshev basis.

    Parameters
    ----------
    xmin, xmax : interval endpoints
    yk         : values f(xk) at the Chebyshev points

    Returns
    -------
    Array of Chebyshev coefficients.
    """
    N = len(yk)
    x = cheb_points(xmin, xmax, N)
    theta = np.arccos((x-(xmax+xmin)/2)*2/(xmax-xmin))
    A = np.array([np.cos(i*theta) for i in range(N)])
    A[1:, :] *= 2
    A /= N
    return np.matmul(A, yk)

In [ ]:
def evaluate_Chebyshev_interpolant(coefficients, x):
    """
    Evaluate a polynomial represented in the Chebyshev basis.

    Parameters
    ----------
    coefficients : Chebyshev coefficients
    x            : evaluation points

    Returns
    -------
    Values of the polynomial at x.
    """
    N = len(coefficients)
    theta = np.arccos(x)
    T = np.cos(np.outer(theta, np.arange(N)))
    return np.matmul(T, coefficients)

1. For the Runge function from Exercise 1 with $n=10$, plot and compare the results produced by this algorithm and by the Lagrange-basis algorithm.

In [ ]:
# Plot the results.
Nx = 1000
xmin = -1
xmax = 1
x = np.linspace(xmin, xmax, Nx)

# Polynomial degree
n = 10

# Chebyshev points
xk_cheb2 = cheb_points(xmin, xmax, n+1)
yk_cheb2 = fRunge(xk_cheb2)

# Interpolation polynomials
coef_cheb2 = compute_Chebyshev_coefficients(-1, 1, yk_cheb2)
p_cheb = Lagrange(xk_cheb2, yk_cheb2, x)
p_cheb2 = evaluate_Chebyshev_interpolant(coef_cheb2, x)

In [ ]:
fig_equi = plt.figure()
plt.scatter(xk_cheb2, yk_cheb2, color="red", label="Data")
plt.plot(x, p_cheb, linestyle="dashed", color="blue", label="P_n(x), Lagrange basis")
plt.plot(x, p_cheb2, linestyle="dotted", color="green", label="P_n(x), Chebyshev basis")
plt.ylim((-0.1, 1.1))
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.show()

2. For the same function, increase the degree beyond $n=660$. Comment on and explain the difference.

In [ ]:
# Plot the results.
Nx = 2000
xmin = -1
xmax = 1
x = np.linspace(xmin, xmax, Nx)

# Polynomial degree
n = 680

# Chebyshev points
xk_cheb2 = cheb_points(xmin, xmax, n+1)
yk_cheb2 = fRunge(xk_cheb2)

# Interpolation polynomials
coef_cheb2 = compute_Chebyshev_coefficients(-1, 1, yk_cheb2)
p_cheb = Lagrange(xk_cheb2, yk_cheb2, x)
p_cheb2 = evaluate_Chebyshev_interpolant(coef_cheb2, x)

In [ ]:
fig_equi = plt.figure()
plt.plot(x, p_cheb, linestyle="dashed", color="blue", label="P_n(x), Lagrange basis")
plt.plot(x, p_cheb2, linestyle="dotted", color="green", label="P_n(x), Chebyshev basis")
plt.ylim((-0.1, 1.1))
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.show()

---

### III. Improving the Lagrange interpolation formula

The algorithm in Part II works very well when both the interpolation points and the polynomial basis are Chebyshev. We now seek more stable algorithms in the Lagrange basis for arbitrary interpolation points.

Recall that

$$
p_n(f)=\sum_{k=0}^nf(x_k)l_k,
$$

and introduce the weights

$$
\omega_k=\frac{1}{\prod_{l\neq k}(x_k-x_l)},
\qquad k=0,\ldots,n.
$$

#### Question 6

Show that

$$
p_n(f)(x)=
\begin{cases}
\displaystyle \Pi_{n+1}(x)\sum_{k=0}^n\frac{\omega_k}{x-x_k}f(x_k), & x\notin\{x_0,\ldots,x_n\},\\
f(x_k), & x=x_k.
\end{cases}
$$

#### Question 7

Implement a function that computes the weights $\omega_k$ from the interpolation points $x_k$, followed by a function that evaluates the interpolation polynomial using the formula above. Test your implementation by checking visually that the polynomial passes through the points $(x_k,y_k)$.


#### Question 8

Approximate the Runge function (1) again using this new algorithm.

*Hint: Gradually increase the number of interpolation points to approximately $n=820$.*

What happens to the coefficients $\omega_k$ as $n$ approaches this value?

### IV. The barycentric interpolation formula

#### Question 9

Show that

$$
\Pi_{n+1}(x)\sum_{k=0}^n\frac{\omega_k}{x-x_k}=1,
$$

and deduce that

$$
p_n(f)(x)=
\begin{cases}
\displaystyle
\frac{\sum_{k=0}^n\frac{\omega_k}{x-x_k}f(x_k)}
     {\sum_{k=0}^n\frac{\omega_k}{x-x_k}}, & x\notin\{x_0,\ldots,x_n\},\\
f(x_k), & x=x_k.
\end{cases}
$$

For Chebyshev interpolation points, assume that

$$
\omega_k=C_n\widetilde\omega_k,
\qquad
\widetilde\omega_k=(-1)^k\sin\left(\frac{(2k+1)\pi}{2n+2}\right).
$$

The common factor $C_n$ cancels in the barycentric formula, giving

$$
p_n(f)(x)=
\begin{cases}
\displaystyle
\frac{\sum_{k=0}^n\frac{\widetilde\omega_k}{x-x_k}f(x_k)}
     {\sum_{k=0}^n\frac{\widetilde\omega_k}{x-x_k}}, & x\notin\{x_0,\ldots,x_n\},\\
f(x_k), & x=x_k.
\end{cases}
$$

#### Question 10

Approximate the Runge function (1) again using the modified barycentric algorithm.

*Hint: Gradually increase the number of interpolation points to approximately $n=5000$.*

### V. Numerical test [2]

Consider

$$
g(x)=\tanh(20\sin(15x))+0.02e^{3x}\sin(300x), \qquad x\in[-1,1]. \tag{3}
$$

We approximate $g$ by its interpolation polynomial at Chebyshev points.

#### Question 11

1. Plot $g$ on $[-1,1]$.

2. Using the algorithms developed above, approximate $g$ by its interpolation polynomial at Chebyshev points, with the largest degree that can be handled reliably.

---

## References

[1] J.-P. Demailly. *Analyse numérique et équations différentielles*, 4th ed. EDP Sciences, 2016.

[2] L. N. Trefethen. *Approximation Theory and Approximation Practice*. SIAM, 2013.

[3] L. N. Trefethen and J. A. C. Weideman. "Two results on polynomial interpolation in equally spaced points." *Journal of Approximation Theory* 65 (1991), 247–260.